# EXP-005 — M4 CPG-GNN: bước 2, TRAIN trên CPG (GPU)

Notebook này train GAT trên CPG đã trích ở bước 1, theo **đúng protocol
M1/M2/M3**: threshold trên validation, test một lần. Input là dataset đã
publish (`semanticcodeclonedetection-m4-cpg-shards`: 52 shard `.jsonl` + 3 npz cache).

**Chạy bằng GPU** (T4/P100) — GAT không train nổi trên CPU. Job dài nên:
chạy tay cell 2→8 trước (toàn bước nhanh, ~15 phút), rồi **Save & Run All**
cho cell 9 (train full). Đừng chạy tay cell 9 rồi giữ tab.

**Trước khi chạy:** Add input dataset shards (tên ở trên). Không còn gì phải sửa.

### Thứ tự chạy
1. Cell 2 (**clone hoặc pull** code mới) → cell 3 (pip torch_geometric)
2. Cell 4 (nạp shards, bung ra JSON) → cell 5 (dựng pairs)
3. Cell 6 PREFLIGHT → cell 7 SMOKE → cell 8 DIAGNOSTIC (gate `logit std > 0`)
4. Cell 9 FULL TRAIN (5 epoch trần, early stopping patience 2 — tự evaluate khi xong)
5. Cell 10 EVALUATE lại + kiểm `EVAL_COMPLETE.json`
6. **Save Version** → gửi `metrics.json` về


In [ ]:
import json
import os
import subprocess
import sys
from pathlib import Path

# Kaggle Add-ons → Secrets → GITHUB_TOKEN + HF_TOKEN (dataset POJ-104 tai tu HF).
GITHUB_TOKEN = ""
HF_TOKEN = ""
try:
    from kaggle_secrets import UserSecretsClient
    _s = UserSecretsClient()
    GITHUB_TOKEN = GITHUB_TOKEN or _s.get_secret("GITHUB_TOKEN")
    HF_TOKEN = HF_TOKEN or _s.get_secret("HF_TOKEN")
except Exception as _e:
    print("[secrets] khong doc duoc secrets:", _e)
if HF_TOKEN:
    os.environ["HF_TOKEN"] = HF_TOKEN
    os.environ["HUGGING_FACE_HUB_TOKEN"] = HF_TOKEN
if GITHUB_TOKEN:
    os.environ.setdefault("GIT_TERMINAL_PROMPT", "0")
print("[secrets] github =", bool(GITHUB_TOKEN), "| hf =", bool(HF_TOKEN))

def sh(cmd, **kw):
    r = subprocess.run(cmd, shell=True, capture_output=True, text=True, **kw)
    print(f"$ {cmd}\n{r.stdout}{r.stderr}")
    if r.returncode != 0:
        raise RuntimeError(f"lenh that bai (rc={r.returncode}): {cmd}")
    return r.stdout


In [ ]:
REPO = "kichirou244/SemanticCodeCloneDetection"
assert REPO != "YOUR_USER/YOUR_REPO", "Sua REPO truoc khi chay cell tiep theo"

if Path(".git").exists():
    # Da clone o session truoc ⇒ **phai pull** — khong pull la chay code cu.
    sh('git pull --ff-only || (git fetch --all --depth 1 && '
       'git reset --hard origin/HEAD)')
else:
    url = f'https://github.com/{REPO}.git'
    if GITHUB_TOKEN:
        url = url.replace('https://', f'https://x-access-token:{GITHUB_TOKEN}@')
    sh(f'git clone --depth 1 {url} repo')
    sh('mv repo/* repo/.[!.]* . 2>/dev/null; rmdir repo')
sh('git log --oneline -1 && python -V && nproc')


In [ ]:
# torch co san tren image GPU Kaggle. Chi cai them torch_geometric (GATConv)
# + transformers (scheduler — giong M1/M2/M3 de so sanh cong bang).
# Neu dong GATConv do: gui nguyen output cell nay ve (sai version torch/cuda).
import torch
print('torch', torch.__version__, '| cuda build:', torch.version.cuda,
      '| gpu:', torch.cuda.is_available(), '|', torch.cuda.get_device_name(0))
sh('pip install -q torch_geometric transformers')
print(sh('python -c "import torch_geometric as pyg; print(\'pyg\', pyg.__version__); '
         'from torch_geometric.nn import GATConv; print(\'GATConv OK\')"'))


In [ ]:
# Nap dataset M4 — chay duoc voi CA dataset cu (folder bung san) LAN dataset
# shards moi. Truoc khi chay: Add input dataset (cu hoac shards deu duoc).
# Input la read-only nen COPY/BUNG ve ./artifacts (benchmark.json can ghi).
# Luu y: KHONG mkdir truoc khi cp -r — mkdir tao thu muc rong lam cp long
# thu muc vao trong (cpg_export/cpg_export/).
import glob
from pathlib import Path
sh('mkdir -p artifacts/cpg_cache')
_exp_path = Path('artifacts/cpg_export')
have = len(list(_exp_path.glob('*.json'))) if _exp_path.exists() else 0
shards = sorted(glob.glob('/kaggle/input/**/cpg_shard_*.jsonl', recursive=True))
print('shards:', len(shards), '| da co:', have)
if have >= 51000:
    print(f'da co ./artifacts ({have} JSON) — dung lai')
elif shards:
    from pathlib import PurePath as _PP
    sdirs = sorted({_PP(s).parent.as_posix() for s in shards})
    npzs = sorted(glob.glob('/kaggle/input/**/cpp_*_full.npz', recursive=True))
    print('bung tu', len(sdirs), 'thu muc shard + copy', len(npzs), 'npz ...')
    sh(f'python scripts/make_cpg_shards.py --from-shards {" ".join(sdirs)} '
       f'--out artifacts/cpg_export')
    for _nz in npzs:
        sh(f'cp {_nz} artifacts/cpg_cache/')
else:
    # Thu muc bung san: duong cua ban uu tien truoc, roi moi glob chung.
    _pref = Path('/kaggle/input/datasets/kichirou/'
                 'semanticcodeclonedetection-dataset-m4/artifacts')
    _exp = _pref if (_pref / 'cpg_export').exists() else None
    if _exp is None:
        _found = glob.glob('/kaggle/input/**/cpg_export', recursive=True)
        _exp = Path(_found[0]).parent if _found else None
    _zp = glob.glob('/kaggle/input/*/*.zip') + glob.glob('/kaggle/input/*.zip')
    sh('rm -rf artifacts/cpg_export artifacts/cpg_cache && mkdir -p artifacts/cpg_cache')
    if _exp is not None:
        print('dung thu muc:', _exp)
        sh(f'cp -r {_exp}/cpg_export artifacts/ && cp -r {_exp}/cpg_cache artifacts/ '
           f'&& echo COPY_XONG')
    elif _zp:
        sh(f'unzip -q -o {_zp[0]} -d _m4in && echo UNZIP_XONG')
        cands = glob.glob('_m4in/**/cpg_export', recursive=True)
        assert cands, 'khong thay cpg_export trong zip'
        src = str(Path(cands[0]).parent)
        sh(f'cp -r {src}/cpg_export artifacts/ && cp -r {src}/cpg_cache artifacts/ '
           f'&& echo COPY_XONG')
    else:
        raise SystemExit('Chua add dataset input M4 — Add input truoc khi chay cell nay')
sh('ls artifacts/cpg_export | wc -l && ls artifacts/cpg_cache/ && du -sh artifacts/')


In [ ]:
# Dung pairs + manifest + samples (can cho validate_dataset va train).
# Build lai bang dung script cua extraction — cung splits, cung leakage check.
# Can HF_TOKEN (cell 1) vi tai POJ-104 tu HuggingFace.
sh('python scripts/build_dataset.py --config configs/pairs/poj104.yaml')
print(sh('python -c "import json; m=json.load(open(\'data/manifests/poj104.json\')); '
         'print(\'leakage_passed =\', m[\'leakage_passed\'])"'))
sh('wc -l data/processed/poj104/samples.jsonl data/pairs/poj104/*.jsonl')


In [ ]:
# PREFLIGHT — gate truoc khi ton gio GPU: schema 6 loai + du export/cache/pairs + VRAM.
sh('python -m src.experiments.exp005_cpg_gnn --config configs/exp005_cpg_gnn.yaml '
   '--mode preflight --device cuda')


In [ ]:
# SMOKE — 64 cap, 1 epoch, vai phut. Dong thoi kiem VRAM voi dung
# batch_size train (batch 16 cap x ~350 node). Do la dung lai.
sh('python -m src.experiments.exp005_cpg_gnn --config configs/exp005_cpg_gnn.yaml '
   '--mode smoke --device cuda')


In [ ]:
# DIAGNOSTIC — grad/logit-std/throughput that. Phai in `logit std > 0`;
# neu scores hoi tu ve 0 thi dung lai. Doc `pairs_per_second_diag` de uoc luong
# gio train: 332.000 cap / toc do nay / 3600 = gio moi epoch.
sh('python -m src.experiments.exp005_cpg_gnn --config configs/exp005_cpg_gnn.yaml '
   '--mode diagnostic --device cuda')


In [ ]:
# FULL TRAIN — 5 epoch tran, early stopping (patience 2) tu dung.
# Train tu goi evaluate khi xong (hetepoch / het budget / early stopping).
# CHAY BANG Save & Run All, dung chay tay giu tab.
TIME_BUDGET_S = 8 * 3600  # limit phien Kaggle 9h tru hao install/dataset
sh('python -m src.experiments.exp005_cpg_gnn --config configs/exp005_cpg_gnn.yaml '
   f'--mode train --device cuda --time-budget-s {TIME_BUDGET_S}')


In [ ]:
# EVALUATE — chay lai tren Kaggle de artifact la nguon so chuan (giong cell 8
# notebook ASTNN). Chay lai nhieu lan an toan: ghi de cung thu muc, khong nhan ban.
sh('python -m src.experiments.exp005_cpg_gnn --config configs/exp005_cpg_gnn.yaml '
   '--mode evaluate --device cuda')
OUT = 'results/exp005_cpg_gnn_poj104'
sh(f'test -f {OUT}/EVAL_COMPLETE.json && echo EVAL_COMPLETE || echo EVAL_CHUA_XONG')
print(sh(f'cat {OUT}/metrics.json'))
print(sh(f'cat {OUT}/threshold.json'))


### Save Version — nhưng **chỉ** khi cell 10 in `EVAL_COMPLETE`

Version đã lưu **là kết quả M4**: `results/exp005_cpg_gnn_poj104` (metrics,
threshold, predictions, checkpoints). Checkpoint giữ lại trong version (vài
trăm MB là cùng — model nhỏ).

**Gửi về 4 thứ:**
1. `metrics.json` (validation + test: P/R/F1/ROC-AUC)
2. `threshold.json`
3. Đuôi `training_history.csv` (val_f1 các epoch — biết đã quá tay hay chưa đủ)
4. `param_count` (in ở log train)

So sánh với M1/M2/M3 sau khi có số — báo cáo sẽ thêm hàng M4 vào bảng kết quả.

### Chưa làm gì ở bước này (cố ý)
- **Chưa tune hyperparameter:** recipe hiện tại là giá trị hợp lý đầu tiên
  (lr 1e-3, batch 16, 5 epoch). Tune sau khi có baseline M4.
- **Chưa chạy nhiều seed:** giống M1/M2 — một seed 42 trước, seed thêm sau.
